# Figure 3 — Encoding models

**a, b** Predictivity of single-feature models, preview versus no preview.
**c, d** Unique predictivity: the drop when one feature is removed from the full
model.

Predictivity is the signed squared Spearman correlation between predicted and
observed held-out log latency, divided by the split-half reliability of that
condition. It is signed because a cross-validated model can do worse than
chance, so negative values are meaningful and are plotted as such. It is not the
coefficient of determination of a fitted model, which is why the axis says
predictivity rather than explained variance.

Single-feature models use all ten predictors. The unique model uses nine: the
untrained CNN is omitted because it is a near-redundant operationalization of
the same quantity as SceneFaceDet (Spearman r = .91), and including both splits
the shared contribution between them.

Error bars are the SD across the 100 folds and are descriptive only — folds are
overlapping resamples of the same images, so all inference is at the image level.
Run `python3 ../code/run_encoding_analyses.py` first to build the cache.

Each table below gives, per condition, predictivity ± SD and `p>0` — the
one-sided permutation test in which the latencies are shuffled across images and
the whole cross-validation is rerun, so the null covers model fitting as well as
scoring — followed by the preview-minus-no-preview difference and its two-sided
test.

Every p is Bonferroni-corrected in the `bonf` column beside it, over its own
family: the >0 tests by 20 (ten features x two conditions) for both models, the
family used in the text and in Supplementary Tables 2 and 3, and the delta tests
by one per feature (10 and 9). Correcting the deltas by the larger factor would
count each feature twice.

In [ ]:
import sys
from pathlib import Path

# The notebooks live next to the code, so the package is importable without
# installing anything. Nothing outside this release is referenced.
sys.path.insert(0, str(Path.cwd().parent / "code"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from facescenes import config as cfg
from facescenes import data as D
from facescenes import behaviour as B
from facescenes import encoding as E
from facescenes import onset_gaze as G
from facescenes import plotting as P
from facescenes import stats as S

P.use_style()
print("stimuli    :", len(D.stimulus_names()))
for t in cfg.TASKS:
    print(f"{t:14s}: {len(D.participant_ids(t))} participants, "
          f"{len(D.analysable_images(t))} images")

In [ ]:
import run_encoding_analyses as RUN

RES = {t: RUN.analyse(t, n_perm=5000) for t in cfg.TASKS}
print({t: len(r["images"]) for t, r in RES.items()}, "images per task")

In [ ]:
def panel(task, kind, fname, alpha=None):
    res = RES[task]
    labels, prev, nop, sd_p, sd_n, pv = [], [], [], [], [], []
    p0_p, p0_n = [], []
    for k, name in enumerate(cfg.FEATURE_NAMES):
        if np.isnan(res[f"{kind}_preview"][k]):
            continue                        # not in the unique model
        labels.append(cfg.FEATURE_LABELS[k])
        prev.append(res[f"{kind}_preview"][k])
        nop.append(res[f"{kind}_no_preview"][k])
        sd_p.append(res[f"{kind}_sd_preview"][k])
        sd_n.append(res[f"{kind}_sd_no_preview"][k])
        pv.append(res[f"p_delta_{kind}"][k])
        # Predictivity > 0: the latencies are shuffled across images and the
        # whole cross-validation is rerun, so the null covers fitting as well as
        # scoring. One-sided -- predicting worse than chance is not of interest.
        p0_p.append(res[f"p_{kind}_preview"][k])
        p0_n.append(res[f"p_{kind}_no_preview"][k])
    prev, nop = np.array(prev), np.array(nop)
    sd_p, sd_n = np.array(sd_p), np.array(sd_n)
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(14, 9))
    ax.bar(x - .2, prev, .4, color=cfg.C_PREVIEW, label="Preview")
    ax.errorbar(x - .2, prev, yerr=P.one_sided_errorbar(prev, sd_p), fmt="None",
                color="k", lw=1.2)
    ax.bar(x + .2, nop, .4, color=cfg.C_NO_PREVIEW, label="No preview")
    ax.errorbar(x + .2, nop, yerr=P.one_sided_errorbar(nop, sd_n), fmt="None",
                color="k", lw=1.2)
    ax.axhline(0, color="k", lw=.8)

    lo = min(0, np.min(np.minimum(prev - sd_p, nop - sd_n)))
    hi = np.max(np.maximum(prev + sd_p, nop + sd_n)) * 1.18
    ticks, (t0, t1) = P.zero_anchored_ticks(lo, hi)
    ax.set_ylim(t0, t1); ax.set_yticks(ticks)
    ax.set_yticklabels([f"{v:g}" for v in ticks], fontsize=P.FONT - 6)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=P.FONT - 6)
    ax.set_ylabel(P.PREDICTIVITY_LABEL if kind == "single" else P.UNIQUE_LABEL,
                  fontsize=P.FONT - 4)
    P.title(ax, cfg.TASK_LABEL[task], fontsize=P.FONT - 2, pad=14)
    if (task, kind) == ("facedetection", "single"):   # one legend serves all panels
        ax.legend(fontsize=P.FONT - 8, frameon=False, loc="upper left")
    P.clean(ax)
    # Predictivity > 0, one-sided, Bonferroni x 20 (ten features x two
    # conditions) for both models, the family used in the text and in
    # Supplementary Tables 2 and 3.
    n_fam = len(cfg.FEATURE_NAMES) * 2
    for xs, vals, sds, ps in ((x - .2, prev, sd_p, p0_p), (x + .2, nop, sd_n, p0_n)):
        for xi, v, s, p in zip(xs, vals, sds, ps):
            # One level only: with 5000 permutations the smallest corrected
            # p is 20/5001 = .004, so finer star levels would be misleading.
            if p * n_fam < .05:
                ax.text(xi, max(v, 0) + s + .01 * (t1 - t0), "*", ha="center",
                        va="bottom", fontsize=P.FONT - 8)
    if alpha is not None:
        for i, p in enumerate(pv):
            if p < alpha:
                top = max(prev[i] + sd_p[i], nop[i] + sd_n[i])
                ax.text(x[i], top + .02 * (t1 - t0), "*", ha="center",
                        va="bottom", fontsize=P.FONT - 4)
    src = pd.DataFrame({
        "feature": labels,
        "preview_r2": prev, "preview_sd_across_folds": sd_p,
        "preview_p": p0_p, "preview_p_bonferroni": np.minimum(1, np.array(p0_p) * n_fam),
        "no_preview_r2": nop, "no_preview_sd_across_folds": sd_n,
        "no_preview_p": p0_n,
        "no_preview_p_bonferroni": np.minimum(1, np.array(p0_n) * n_fam)})
    P.save(fig, fname, src); plt.show()

    # Two families, two factors: the >0 tests are one per feature x condition,
    # the delta tests one per feature.
    n_p0, n_delta = len(cfg.FEATURE_NAMES) * 2, len(labels)
    print(f"{cfg.TASK_LABEL[task]}  —  {kind} predictivity")
    print(f"  {'feature':14s} {'preview':>16s} {'p>0':>7s} {'bonf':>6s}  "
          f"{'no preview':>16s} {'p>0':>7s} {'bonf':>6s}  "
          f"{'delta':>8s} {'p':>7s} {'bonf':>6s}")
    for i, lab in enumerate(labels):
        print(f"  {lab:14s} {prev[i]:+8.3f}+/-{sd_p[i]:.3f} "
              f"{p0_p[i]:7.4f} {min(1, p0_p[i]*n_p0):6.3f}  "
              f"{nop[i]:+8.3f}+/-{sd_n[i]:.3f} "
              f"{p0_n[i]:7.4f} {min(1, p0_n[i]*n_p0):6.3f}  "
              f"{prev[i]-nop[i]:+8.3f} {pv[i]:7.4f} {min(1, pv[i]*n_delta):6.3f}")
    print(f"  p>0 one-sided against shuffled images, Bonferroni x{n_p0} "
          f"({n_delta} features x 2 conditions); delta two-sided, "
          f"Bonferroni x{n_delta} (one test per feature); "
          f"smallest attainable p {1/5001:.1e}")


panel("facedetection", "single", "fig3a_single_face_detection.png")

In [ ]:
panel("movingwindow", "single", "fig3b_single_moving_window.png")

In [ ]:
panel("facedetection", "unique", "fig3c_unique_face_detection.png")

In [ ]:
panel("movingwindow", "unique", "fig3d_unique_moving_window.png")

## Standardised regression coefficients

Predictors and the dependent variable are z-scored within each training fold, so
β is the change in log latency in SD units per 1 SD of the predictor. A negative
β means the feature predicts *faster* detection. β is not noise-corrected: the
reliability correction rescales predictivity, it does not apply to a slope.

In [ ]:
for task in cfg.TASKS:
    res = RES[task]
    print(cfg.TASK_LABEL[task])
    print(f"  {'feature':14s} {'single-feature model':>26s} {'full 9-feature model':>26s}")
    print(f"  {'':14s} {'preview':>12s} {'no prev':>13s} {'preview':>12s} {'no prev':>13s}")
    for k, lab in enumerate(cfg.FEATURE_LABELS):
        f_p, f_n = res["beta_full_preview"][k], res["beta_full_no_preview"][k]
        full = ("        --      --   " if np.isnan(f_p)
                else f"{f_p:+6.3f}+/-{res['beta_full_sd_preview'][k]:.3f} "
                     f"{f_n:+6.3f}+/-{res['beta_full_sd_no_preview'][k]:.3f}")
        print(f"  {lab:14s} "
              f"{res['beta_single_preview'][k]:+6.3f}+/-{res['beta_single_sd_preview'][k]:.3f} "
              f"{res['beta_single_no_preview'][k]:+6.3f}+/-{res['beta_single_sd_no_preview'][k]:.3f} "
              f"{full}")
    print()